# Cell segmentation using Cellpose

## Import packages

We import the necessary packages. The `ISS_postprocessing` module uses **Cellpose**, which can take advantage of a **CUDA-compatible GPU** for faster segmentation.

A GPU is **recommended**, especially for large images, but the code will still run on **CPU** if no GPU is available (with slower performance).


In [ ]:
import ISS_postprocessing.segmentation as SEG

## Cell segmentation using Cellpose

In this step we build a **segmentation mask** from the DAPI signal using **Cellpose**.

The segmentation function supports two input modes:

**1. Retiled mode** (`input_image_type="retiled"`)

All DAPI tiles in

```
/preprocessing/Cycle1/4_retiled/
```

(e.g. `Cycle1_s0_ch4.tif`, `Cycle1_s1_ch4.tif`, …) are segmented individually.  
Tile positions from

```
/preprocessing/Cycle1/4_retiled/Cycle1_retiled_coords.csv
```

are then used to stitch the tiles into one full segmentation mask:

```
{region}_cellpose_retiled_expanded.npz
```

**2. Stitched mode** (`input_image_type="stitched"`)

Cellpose runs directly on the stitched image:

```
/preprocessing/Cycle1/3_stitched/Cycle1_ch4.tif
```

The result is saved as:

```
{region}_cellpose_stitched_expanded.npz
```

**Retiled mode is recommended for very large images that may exceed GPU memory when processed as a single stitched image.**


### Function parameters

`input_dir` (str) 
Path to the parent directory containing the preprocessed region folders (e.g. `/R1/`, `/R2/`, …). These region folders are created automatically during preprocessing.

`region` (str) 
Region identifier to process (e.g. `"R1"`).

`input_image_type` ("retiled" | "stitched", default="retiled") 
Selects which image layout to use for segmentation.

- `"retiled"`: segment all tiles in `4_retiled` and stitch the results
- `"stitched"`: segment the stitched image in `3_stitched`

`output_dir_prefix` (str | None, default=None) 
Optional base directory where segmentation outputs should be written.

- If `None`, outputs are written inside each region directory:

```
input_dir/R#/postprocessing/segmentation/
```

- If set, outputs are written to:

```
output_dir_prefix/R#/postprocessing/segmentation/
```

This is useful when running on clusters or shared servers where results should be written to a scratch directory.

`DAPI_ch` (int, default=4)
Index of the DAPI channel used for segmentation.

Example filenames:

```
Cycle1_s0_ch4.tif
Cycle1_ch4.tif
```

`diameter` (float | int | None)
Approximate object diameter for Cellpose (in pixels). If not specified, Cellpose estimates the diameter automatically.

`expanded_distance` (int, default=20)  
Number of pixels by which each segmented nucleus is expanded after watershed refinement.

---


> **Note:**  
> You need to specify the names of the regions you want to post-process.  
> Multiple regions can be processed in one run, and they will all be analyzed using the same parameters.  
> In most cases this is fine, but there may be situations where individual regions need to be treated differently.  


In [ ]:
input_dir = '/path/to/regions/'
regions = ['R1', 'R2', 'R3']

In [ ]:
for region in regions:
        
    SEG.cell_pose_segmentation_to_coo(
        input_dir,
        region,
        output_dir_prefix = None,     # or '/path/to/preferred/output/dir'
        DAPI_ch=4,
        input_image_type='stitched',  # or 'retiled'
        diameter=None,
        expanded_distance=20
        )
    
